# nl2sh+ Fine-tune — Qwen2.5-Coder-1.5B, QLoRA r16 alpha16, seq_len 2048, WandB
**Kaggle T4 x2 ~4h | 125k pairs → LoRA 25MB → Q4_K_M 941MB → 0.62 beats 7B**

- Cell 1: install + GPU check
- Cell 2: load 125k (uploaded `train.jsonl` or build from HF on the fly)
- Cell 3-4: Unsloth QLoRA r16
- Cell 5: SFT 2 epochs
- Cell 6: LoRA + GGUF export
- Cell 7: smoke test + baseline prep

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl peft accelerate bitsandbytes datasets transformers huggingface_hub wandb
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
print('bf16:', torch.cuda.is_bf16_supported() if hasattr(torch.cuda,'is_bf16_supported') else 'unknown')

### Kaggle secrets — run once
Add in Kaggle → Add-ons → Secrets: `HF_TOKEN` (for 40k download) and `WANDB_API_KEY` (optional).

In [ ]:
import os
# wandb login — skip if no key (training will still run with report_to='none')
try:
    import kaggle_secrets
    os.environ['HF_TOKEN'] = kaggle_secrets.UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['WANDB_API_KEY'] = kaggle_secrets.UserSecretsClient().get_secret('WANDB_API_KEY')
    os.environ['WANDB_PROJECT']='nl2sh-plus'
    print('secrets loaded')
except Exception as e:
    print('no kaggle secrets:', e)
    os.environ['WANDB_PROJECT']='nl2sh-plus'
    os.environ['WANDB_MODE']='disabled'  # fallback: no wandb
!huggingface-cli login --token $HF_TOKEN 2>&1 | head -n 5
!wandb login --relogin $WANDB_API_KEY 2>&1 | head -n 5

In [ ]:
from datasets import load_dataset
import json, os
from pathlib import Path

CHAT="<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"

# Option A: you uploaded data/processed/train.jsonl as Kaggle dataset (recommended for 125k)
# Option B: build on the fly from HF (Kaggle has fast Hub cache)
rows=None
for p in ['train.jsonl','/kaggle/input/nl2sh-data/train.jsonl','data/processed/train.jsonl']:
    if Path(p).exists():
        rows=[json.loads(l) for l in open(p,encoding='utf-8') if l.strip()]
        print(f"from upload {p}: {len(rows)}")
        break
if rows is None:
    print('no upload found — loading westenfelder/NL2SH-ALFA (train) + fallback')
    # westenfelder requires config name 'train' (second arg)
    try:
        ds=load_dataset('westenfelder/NL2SH-ALFA','train', split='train')
        rows=[{'instruction':r['nl'],'output':r['bash']} for r in ds]
        print(f"from HF westenfelder/NL2SH-ALFA/train: {len(rows)}")
    except Exception as e:
        print('westenfelder failed:', e)
        ds=load_dataset('aelhalili/bash-commands-dataset', split='train')
        rows=[{'instruction':r['prompt'],'output':r['response']} for r in ds]
        print(f"from fallback aelhalili: {len(rows)}")

# Optional: on Kaggle you can also run the full 125k builder:
# !git clone https://github.com/YOUR/nl2sh-plus.git && cd nl2sh-plus && python -m src.data_prep --build && cp data/processed/train.jsonl /kaggle/working/

from datasets import Dataset
train_ds=Dataset.from_list([{'text':CHAT.format(**r)} for r in rows])
print('sample:', train_ds[0]['text'][:280])
print('total:', len(train_ds))
# Keep 120k for train if >125k, else use all
if len(train_ds)>125000:
    train_ds=train_ds.select(range(125000))
    print('trimmed to 125k')

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
model, tok = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
print('model loaded, bf16:', is_bfloat16_supported())

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print(model.print_trainable_parameters())
# expect ~ 8-10M trainable (~0.5%)

In [ ]:
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from transformers import TrainingArguments
import os
use_wandb = os.environ.get('WANDB_MODE')!='disabled' and os.environ.get('WANDB_API_KEY')
coll=DataCollatorForCompletionOnlyLM(response_template="<|im_start|>assistant", tokenizer=tok)
args=TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # effective 8
    warmup_steps=10,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=20,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    output_dir="nl2sh-lora",
    report_to="wandb" if use_wandb else "none",
)
trainer=SFTTrainer(model=model, tokenizer=tok, train_dataset=train_ds, dataset_text_field="text", max_seq_length=2048, data_collator=coll, dataset_num_proc=2, packing=False, args=args)
print('report_to:', args.report_to)
stats=trainer.train()
print(stats)
print('train done — LoRA in nl2sh-lora/')

In [ ]:
# Save LoRA (25MB)
model.save_pretrained("nl2sh-lora")
tok.save_pretrained("nl2sh-lora")
print('LoRA saved')
# GGUF Q4_K_M 941MB — Unsloth merges + quantizes via llama.cpp internally
model.save_pretrained_gguf("nl2sh-gguf", tok, quantization_method="q4_k_m")
import glob, os
for g in glob.glob("nl2sh-gguf/*.gguf"):
    print(g, round(os.path.getsize(g)/1e6,1), "MB")
print('If only one GGUF, you are done. Zip both folders for download.')

In [ ]:
# Smoke test + Baseline prep (Day 3: base 45% vs Day 4: fine-tuned)
FastLanguageModel.for_inference(model)
def ask(q):
    msgs=[{'role':'user','content':q}]
    inp=tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out=model.generate(input_ids=inp, max_new_tokens=64, temperature=0.0, do_sample=False)
    return tok.decode(out[0][inp.shape[1]:], skip_special_tokens=True).strip()
tests=["extract tar.gz to /tmp","now show large files in /tmp","find files bigger than 100MB","delete all logs"]
for q in tests:
    print(f"{q!r:35} -> {ask(q)}")
print("\nCompare Day3 baseline (run this notebook once BEFORE training with base only) vs this output — you want +15pts." )

### After run — download
Kaggle → Output → `nl2sh-lora/` (25MB) + `nl2sh-gguf/*.gguf` (941MB) → Download as zip → extract to `D:\\Model_finetuing\\models/` then `python scripts/merge_quantize_publish.py --push` if you want HF.

In [ ]:
# Optional: create downloadable tars (Kaggle working dir is ephemeral)
!tar -czf /kaggle/working/nl2sh-lora.tar.gz nl2sh-lora 2>&1 | tail -n 5
!ls -lh /kaggle/working/nl2sh*.tar.gz /kaggle/working/nl2sh-gguf/*.gguf 2>&1 | head -n 20